# Installation README

## For this particular project to MUST have a local virtual environment!!!

The requirements are  listed in clean_env_requirements.txt

### Adding the SRC files:
In the main folder are 2 files: requirements.txt, setup.py
setup.py: allows you to add folders as packages (that's why you need a virtual environment). To install these packages, once you've activated the virtual environment use "pip install -e ." from the main folder (where setup.py is located).
requirements.txt: install (to the virtual environment) using "pip install -r requirements.txt" 

# Importing 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats
from src.altering_plate_multiple_collection_methods import *
from src.bridging_coefficients import *
from src.distribution_assessment import *
from src.adat_handling import read_adat_file
from src.dataframe_transformation import *
from src.prophet_scores import * 
from modeling import rap_response_predictor


# Getting the data

In [ ]:
# # get base RFUs from Izhar's parquet files
# base_rfus = pd.read_parquet('../../data/adat/base/v_175_OH2026_013.parquet').reset_index(drop=False)
# info_df = pd.read_csv('../../data/excels/20260406_adat_data_samples_with_sample_info.csv')


In [ ]:
base_rfus_path = '../../data/adat/base/v_175_OH2026_013.parquet'
info_path = '../../data/excels/20260406_adat_data_samples_with_sample_info.csv'

In [ ]:
def get_base_rfu_data(base_rfus_path, info_path):
    
    
    #Reading the data
    base_rfus = pd.read_parquet(base_rfus_path).reset_index(drop=False)
    info_df = pd.read_csv(info_path)

    #Cleaning PlateId values
    base_rfus['PlateId'] = base_rfus['PlateId'].replace('OH2023_001', 'OH2023_01')
    info_df['PlateId'] = info_df['PlateId'].str.split('_set').str[0]
    base_rfus['PlateId'] = base_rfus['PlateId'].str.split('_set').str[0]
    base_rfus['PlateId'] = base_rfus['PlateId'].str.split('_Lot3').str[0]
    base_rfus = add_measure_id(base_rfus)
    
    #Filtering
    edta_info_df = filter_dataframe(info_df, info_df['SampleMatrixType']=='EDTA Plasma',
                                info_df['SubType']=='PROPHET')
    edta_rfu_df = base_rfus[base_rfus['MeasureId'].isin(edta_info_df['MeasureId'])].drop_duplicates(subset=['MeasureId']).reset_index(drop=True)
    
    ds_info_df = filter_dataframe(info_df, info_df['SampleMatrixType']=='Streck DS2',
                                info_df['SubType']=='PROPHET')
    ds_rfu_df = base_rfus[base_rfus['MeasureId'].isin(ds_info_df['MeasureId'])].drop_duplicates(subset=['MeasureId']).reset_index(drop=True)

    nds_info_df = filter_dataframe(info_df, info_df['SampleMatrixType']=='Streck non-DS')
    nds_rfu_df = base_rfus[base_rfus['MeasureId'].isin(nds_info_df['MeasureId'])].drop_duplicates(subset=['MeasureId']).reset_index(drop=True)




    return base_rfus, info_df, edta_rfu_df, ds_rfu_df, nds_rfu_df

In [ ]:
base_rfus, info_df, edta_rfu_df, ds_rfu_df, nds_rfu_df = get_base_rfu_data(base_rfus_path, info_path)

In [ ]:
info_df['SampleMatrixType'].value_counts()

In [ ]:
edta_rfu_df[edta_rfu_df['PlateId'].str.startswith('OH2025')]['PlateId'].str.split('_').str[1].unique()

In [ ]:
ds_rfu_df[ds_rfu_df['PlateId']=='OH2026_013']

In [ ]:
nds_rfu_df['PlateId'].value_counts()

In [ ]:
plates_051_055 = [f'OH2025_05{i}' for i in range(1, 6)]

In [ ]:
ds_rfu_051_055_df = ds_rfu_df[ds_rfu_df['PlateId'].isin(plates_051_055)]

# Creating (or loading) bridging factors

In [ ]:
plates = [f"OH2025_05{i}" for i in range(1, 8)] 
plates += [f"OH2026_00{i}" for i in range(1, 4)]

In [ ]:
ds_rfu_df = ds_rfu_df[ds_rfu_df['PlateId'].isin(plates)]

In [ ]:
nds_rfu_df

In [ ]:
ds_rfu_df

In [ ]:
aptamers = get_aptamers(ds_rfu_df)

In [ ]:
params_ds_051_003 = fit_mad_log_transform(ds_rfu_df, edta_rfu_df, aptamers)

In [ ]:
params_nds_051_003 = fit_mad_log_transform(nds_rfu_df, edta_rfu_df, aptamers)

In [ ]:
params_nds_051_003

In [ ]:
os.getcwd()

In [ ]:
# import json
# with open('../../data/params/pre_hybridization/ds_log_04032026.json', 'w') as fp:
#     json.dump(params_ds_051_003, fp)
# with open('../../data/params/pre_hybridization/nds_log_04032026.json', 'w') as fp:
#     json.dump(params_nds_051_003, fp)

# Transform plates

In [ ]:
from src.altering_plate_multiple_collection_methods import alter_scanner_files_MAD_bridging as alter_multiple_mad
# from src.altering_plate_multiple_collection_methods import alter_scanner_files_median_bridging as alter_multiple_median

In [ ]:
plates = [f"OH2025_05{i}" for i in range(1, 8)]
plates += [f"OH2026_00{i}" for i in range(1, 10)]

In [ ]:
plates

In [ ]:
for plate in plates:
    alter_multiple_mad(f'../../data/plates/original_with_altered_workbooks/{plate}',
            f'../../data/plates/original_with_altered_workbooks/{plate}/{plate} Workbook.xlsx',
            
            {'Streck': '../../data/params/pre_normalization/ds_log_04032026.json',
            'NDS': '../../data/params/pre_normalization/nds_log_04032026.json'},
            f'results/plates/median_transformed_no_hces_26032026/{plate}')

In [ ]:
for plate in plates:
    alter_multiple_median(f'../../data/plates/original_with_altered_workbooks/{plate}',
            f'../../data/plates/original_with_altered_workbooks/{plate}/{plate} Workbook.xlsx',
            
            {'Streck': '../../data/params/pre_hybridization/ds_log_04032026.json',
            'NDS': '../../data/params/pre_hybridization/nds_log_04032026.json'},
            f'results/plates/median_transformed_09032026/{plate}')

# Flags

In [ ]:
from pathlib import Path

def concat_final_dfs(path):
    folder = Path(path)

    dfs = []

    for file in folder.rglob("*.adat"):  # search recursively in subfolders
        if "anmlSMP.adat" in file.name:
            df, _ = read_adat_file(str(file))
            dfs.append(df)
    # print(len(dfs))
    df = pd.concat(dfs, ignore_index=True)
    return df

    

In [ ]:
import re

def concat_initial_dfs(path):
    folder = Path(path)

    dfs = []

    for file in folder.rglob("*.adat"):
        if re.search(r"\d\.adat$", file.name):
            df, _ = read_adat_file(str(file))
            dfs.append(df)

    df = pd.concat(dfs, ignore_index=True)
    return df

In [ ]:

def flag_report(bridged_adat_path, unbridged_adat_path, ds_rfu_df, nds_rfu_df):

    bridged_df = concat_final_dfs(bridged_adat_path)
    unbridged_df = concat_final_dfs(unbridged_adat_path)
    
    bridged_df['MeasureId'] = bridged_df['PlateId'] + '_' + bridged_df['PlatePosition']
    unbridged_df['MeasureId'] = unbridged_df['PlateId'] + '_' + unbridged_df['PlatePosition']
    
    ds_bridged_df = bridged_df[bridged_df['MeasureId'].isin(ds_rfu_df['MeasureId'])]
    ds_unbridged_df = unbridged_df[unbridged_df['MeasureId'].isin(ds_rfu_df['MeasureId'])]

    nds_bridged_df = bridged_df[bridged_df['MeasureId'].isin(nds_rfu_df['MeasureId'])]
    nds_unbridged_df = unbridged_df[unbridged_df['MeasureId'].isin(nds_rfu_df['MeasureId'])]
    
    print("DS unbridged flags:")
    print(ds_unbridged_df['RowCheck'].value_counts())
    print("DS bridged flags:")
    print(ds_bridged_df['RowCheck'].value_counts())
    print("nDS unbridged flags:")
    print(nds_unbridged_df['RowCheck'].value_counts())
    print("nDS bridged flags:")
    print(nds_bridged_df['RowCheck'].value_counts())
    
    return ds_unbridged_df, ds_bridged_df, nds_unbridged_df, nds_bridged_df

In [ ]:
bridged_adat_path = 'results/adats/madmed_09032026'
unbridged_adat_path = 'results/adats/no_bridging'

In [ ]:
ds_unbridged_df, ds_bridged_df, nds_unbridged_df, nds_bridged_df = flag_report(bridged_adat_path, unbridged_adat_path, ds_rfu_df, nds_rfu_df)

In [ ]:
path1 = 'results/adats/madmed_no_hces_26032026'

In [ ]:
os.listdir(path1)

In [ ]:
initial_df = concat_initial_dfs(path1)

In [ ]:
initial_df = add_measure_id(initial_df)

In [ ]:
initial_df[initial_df['MeasureId'].isin(nds_unbridged_df['MeasureId'])]

In [ ]:
ds_unbridged_df['PlateId'].value_counts()

In [ ]:
model = pd.read_pickle('../../data/models/DCB_nosex_gamma1_rescaled.pkl')

In [ ]:
from src.prophet_scores import get_prophet_score
def add_prophet_score(df, model):
    df1 = df.copy()
    df1['PROphetScore'] = get_prophet_score(df1, model)
    return df1

In [ ]:
def overlay_histograms(series_list, names, bins=50, alpha=0.4):
    """
    Plot normalized histograms of multiple pandas Series on the same axes.

    Parameters
    ----------
    series_list : list of pandas.Series
        The data series to plot.
    names : list of str
        Names corresponding to each series.
    bins : int or sequence
        Number of bins or bin edges.
    alpha : float
        Transparency for overlay.
    """

    plt.figure()

    for s, name in zip(series_list, names):
        values = s.dropna().values
        plt.hist(values, bins=bins, density=True, alpha=alpha, label=name)

    plt.xlabel("Value")
    plt.ylabel("Frequency density")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:

from __future__ import annotations

from typing import Sequence, Optional, Union
import numpy as np
import matplotlib.pyplot as plt

ArrayLike = Union[Sequence[float], np.ndarray]

def plot_density_histograms(
    series_list: Sequence[ArrayLike],
    names: Optional[Sequence[str]] = None,
    title: str = "",
    bins: Union[int, Sequence[float], str] = "auto",
    alpha: Optional[float] = None,
    histtype: Optional[str] = None,
    common_range: bool = True,
    dropna: bool = True,
    trim: Optional[float] = None,  # NOW: percent to KEEP (e.g. 0.95 keeps central 95%)
    ax: Optional[plt.Axes] = None,
    linewidth: Optional[float] = None,
    ylabel: str = "Density",
    xlabel: str = "PROphet Score",
):
    """
    Plot multiple density histograms on the same axes.

    Parameters
    ----------
    trim : float in (0, 1], optional
        Fraction of data to KEEP from the center (symmetric tails removed).
        Example: trim=0.95 keeps the central 95% => removes 2.5% from each tail.
        trim=1.0 keeps everything (no trimming).
    """

    if not series_list:
        raise ValueError("series_list is empty.")

    n = len(series_list)

    # Auto styling for readability with many overlays
    if histtype is None:
        histtype = "step" if n > 3 else "stepfilled"
    if alpha is None:
        alpha = 1.0 if histtype == "step" else 0.35
    if linewidth is None:
        linewidth = 2.0 if histtype == "step" else 1.0

    # Validate trim (percent to keep)
    if trim is not None:
        if not (0 < trim <= 1):
            raise ValueError("trim must be in (0, 1]. Example: trim=0.95 keeps central 95%.")
        tail = (1.0 - float(trim)) / 2.0
    else:
        tail = None

    # Convert, clean, trim
    cleaned = []
    auto_names = []

    for i, s in enumerate(series_list, start=1):
        arr = np.asarray(s, dtype=float).ravel()

        if dropna:
            arr = arr[np.isfinite(arr)]

        if tail is not None and tail > 0 and arr.size > 0:
            lower = np.quantile(arr, tail)
            upper = np.quantile(arr, 1.0 - tail)
            arr = arr[(arr >= lower) & (arr <= upper)]

        cleaned.append(arr)

        series_name = getattr(s, "name", None)
        auto_names.append(series_name if series_name else f"Series {i}")

    if names is None:
        names = auto_names
    if len(names) != len(cleaned):
        raise ValueError("Length of 'names' must match length of 'series_list'.")

    # Axes setup
    if ax is None:
        fig, ax = plt.subplots(figsize=(9, 5))
    else:
        fig = ax.figure

    # Shared range
    hist_range = None
    if common_range:
        mins = [a.min() for a in cleaned if a.size]
        maxs = [a.max() for a in cleaned if a.size]
        if mins and maxs:
            hist_range = (min(mins), max(maxs))

    # High-contrast colors + linestyles (helps when 4+ series)
    cmap = plt.get_cmap("tab10")
    colors = [cmap(i % 10) for i in range(n)]
    linestyles = ["-", "--", "-.", ":"]  # cycles nicely

    for idx, (arr, label) in enumerate(zip(cleaned, names)):
        if arr.size == 0:
            continue

        color = colors[idx]
        ls = linestyles[idx % len(linestyles)]

        ax.hist(
            arr,
            bins=bins,
            range=hist_range,
            density=True,
            histtype=histtype,
            alpha=alpha,
            linewidth=linewidth,
            linestyle=ls if histtype == "step" else "-",  # linestyle mainly matters for step
            color=color,
            label=label,
        )

    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xlabel(xlabel)
    ax.legend()
    ax.grid(True, alpha=0.25)

    return fig, ax

In [ ]:
ds_unbridged_df['PROphetScore'] = get_prophet_score(ds_unbridged_df, model)
ds_bridged_df['PROphetScore'] = get_prophet_score(ds_bridged_df, model)
nds_unbridged_df['PROphetScore'] = get_prophet_score(nds_unbridged_df, model)
nds_bridged_df['PROphetScore'] = get_prophet_score(nds_bridged_df, model)

In [ ]:
plot_density_histograms([ds_unbridged_df['PROphetScore'], ds_bridged_df['PROphetScore'], nds_unbridged_df['PROphetScore'], nds_bridged_df['PROphetScore']],
                 ['DS Unbridged', 'DS Bridged', 'nDS Unbridged', 'nDS Bridged'])

# COMPLEMENTARY(!!!) Post-ANML bridging

In [ ]:
final_rfus = pd.read_parquet('../../data/adat/anmlSMP/v_249_OH2024_036.parquet').reset_index(drop=False)

In [ ]:
final_rfus = add_measure_id(final_rfus)

In [ ]:
final_edta_rfus = final_rfus[final_rfus['MeasureId'].isin(edta_rfu_df['MeasureId'])].drop_duplicates(subset=['MeasureId'])

In [ ]:
ds_bridged_postANML_coefficients = fit_mad_linear_transform(ds_bridged_df, final_edta_rfus, aptamers)
nds_bridged_postANML_coefficients = fit_mad_linear_transform(nds_bridged_df, final_edta_rfus, aptamers)
ds_unbridged_postANML_coefficients = fit_mad_linear_transform(ds_unbridged_df, final_edta_rfus, aptamers)
nds_unbridged_postANML_coefficients = fit_mad_linear_transform(nds_unbridged_df, final_edta_rfus, aptamers)



In [ ]:
ds_bridged_postANML_coefficients['10000-28']

In [ ]:
nds_bridged_postANML_coefficients['10000-28']

In [ ]:
ds_unbridged_postANML_coefficients['10000-28']

In [ ]:
nds_unbridged_postANML_coefficients['10000-28']

In [ ]:
# import json
# with open('../../data/params/post_hybridization/ds_med_08032026.json', 'w') as fp:
#     json.dump(ds_bridged_postANML_coefficients, fp)
# with open('../../data/params/post_hybridization/nds_med_08032026.json', 'w') as fp:
#     json.dump(nds_bridged_postANML_coefficients, fp)
# with open('../../data/params/post_hybridization/ds_unbridged_med_08032026.json', 'w') as fp:
#     json.dump(ds_unbridged_postANML_coefficients, fp)
# with open('../../data/params/post_hybridization/nds_unbridged_med_08032026.json', 'w') as fp:
#     json.dump(nds_unbridged_postANML_coefficients, fp)

In [ ]:
from src.bridging import transform_median_post_anml

In [ ]:
ds_bridged_postANML = transform_median_post_anml(ds_bridged_df, ds_bridged_postANML_coefficients)
nds_bridged_postANML = transform_median_post_anml(nds_bridged_df, nds_bridged_postANML_coefficients)
ds_unbridged_postANML = transform_median_post_anml(ds_unbridged_df, ds_unbridged_postANML_coefficients)
nds_unbridged_postANML = transform_median_post_anml(nds_unbridged_df, nds_unbridged_postANML_coefficients)



In [ ]:
nds_bridged_postANML[aptamers]/nds_bridged_df[aptamers]

In [ ]:
ds_bridged_postANML['PROphetScore'] = get_prophet_score(ds_bridged_postANML, model)
nds_bridged_postANML['PROphetScore'] = get_prophet_score(nds_bridged_postANML, model)
ds_unbridged_postANML['PROphetScore'] = get_prophet_score(ds_unbridged_postANML, model)
nds_unbridged_postANML['PROphetScore'] = get_prophet_score(nds_unbridged_postANML, model)

In [ ]:
plot_density_histograms([ds_unbridged_postANML['PROphetScore'], ds_bridged_postANML['PROphetScore'], nds_unbridged_postANML['PROphetScore'], nds_bridged_postANML['PROphetScore']],
                 ['DS Unbridged', 'DS Bridged', 'nDS Unbridged', 'nDS Bridged'], title="Post-ANML median bridging PROphet Scores", bins=20)

In [ ]:
plot_density_histograms([ds_bridged_postANML['PROphetScore'], nds_bridged_postANML['PROphetScore']],
                 ['DS Bridged', 'nDS Bridged'], title="Post-ANML median bridging PROphet Scores", bins=20)

In [ ]:
plot_density_histograms([ds_bridged_postANML['PROphetScore']],
                 ['DS Bridged'], title="Post-ANML median bridging PROphet Scores", bins=20)

In [ ]:
(ds_bridged_postANML['PROphetScore']>=5).sum()/len(ds_bridged_postANML['PROphetScore'])

In [ ]:
(nds_bridged_postANML['PROphetScore']>=5).sum()/len(nds_bridged_postANML['PROphetScore'])

# Spearman comparisons

## 1. Initial

In [ ]:
bridged_adat_path = 'results/adats/madmed_09032026'
unbridged_adat_path = 'results/adats/no_bridging'

In [ ]:
initial_unbridged = concat_initial_dfs(unbridged_adat_path)
initial_bridged = concat_initial_dfs(bridged_adat_path)

In [ ]:
initial_unbridged

In [ ]:
initial_bridged

In [ ]:
initial_bridged = add_measure_id(initial_bridged)
initial_unbridged = add_measure_id(initial_unbridged)

In [ ]:
initial_bridged_ds = initial_bridged[initial_bridged['MeasureId'].isin(ds_rfu_df['MeasureId'])]
initial_bridged_nds = initial_bridged[initial_bridged['MeasureId'].isin(nds_rfu_df['MeasureId'])]
initial_unbridged_ds = initial_unbridged[initial_unbridged['MeasureId'].isin(ds_rfu_df['MeasureId'])]
initial_unbridged_nds = initial_unbridged[initial_unbridged['MeasureId'].isin(nds_rfu_df['MeasureId'])]

In [ ]:
initial_bridged_ds[aptamers]

In [ ]:
initial_unbridged_ds[['MeasureId'] + aptamers]

In [ ]:
initial_bridged_ds['10001-7']/initial_unbridged_ds['10001-7']

In [ ]:
spearman = initial_unbridged_ds[aptamers].corrwith(initial_bridged_ds[aptamers], method="spearman")
print(spearman[spearman < 0.999])

In [ ]:
assert np.allclose(
    initial_unbridged_ds[aptamers].corrwith(initial_bridged_ds[aptamers], method="spearman"),
    1,
    atol=1e-3
)

## 2. Final

In [ ]:
assert np.allclose(
    ds_bridged_df[aptamers].corrwith(ds_bridged_postANML[aptamers], method="spearman"),
    1,
    atol=1e-12
)

In [ ]:
spearman = ds_bridged_df[aptamers].corrwith(ds_bridged_postANML[aptamers], method="spearman")
print(spearman[spearman < 0.999999])

In [ ]:
ds_bridged_postANML['10000-28'].corr(ds_bridged_df['10000-28'], method="spearman")

## 3. Initial vs Final

In [ ]:
spearman = initial_bridged_ds[aptamers].corrwith(ds_bridged_postANML[aptamers], method="spearman")
print(spearman[spearman < 0.999999])

In [ ]:
spearman.hist(bins=100, density=True)
plt.xlabel("Spearman correlation coefficient")
plt.ylabel("Frequency density")
plt.title("Distribution of Spearman correlations between initial and post-ANML, all aptamers")


In [ ]:
raps = [key for key in model.RAPs.keys()]

In [ ]:
spearman[raps].hist(bins=100, density=True)
plt.xlabel("Spearman correlation coefficient, RAPs only")
plt.ylabel("Frequency density")
plt.title("Distribution of Spearman correlations between initial and post-ANML, RAPs only")

# Matched Samples

In [ ]:
ds_bridged_df['PROphetScore']

In [ ]:
nds_bridged_df['PROphetScore']

In [ ]:
ds_bridged_postANML['PROphetScore']

In [ ]:
matched_samples = pd.read_excel('../../data/excels/all_matched_samples_streck.xlsx')

In [ ]:
matched_samples

In [ ]:
matched_edta = matched_samples[matched_samples['SampleRun'] == 'EDTA']
matched_ds = matched_samples[matched_samples['SampleRun'] == 'DS']
matched_nds = matched_samples[matched_samples['SampleRun'] == 'non DS']

In [ ]:
matched_edta

In [ ]:
matched_samples['SubjectId'].value_counts()

In [ ]:
matched_edta_ds = pd.merge(matched_edta, matched_ds, how='inner', on='SubjectId', suffixes=('_edta', '_ds'))

In [ ]:
matched_edta_ds

In [ ]:
matched_edta_nds = pd.merge(matched_edta, matched_nds, how='inner', on='SubjectId', suffixes=('_edta', '_nds'))

In [ ]:
matched_edta_nds.shape

In [ ]:
matched_edta_nds[matched_edta_nds['MeasureId_nds'].isin(initial_unbridged_nds['MeasureId'])]

In [ ]:
matched_edta_ds[matched_edta_ds['MeasureId_ds'].isin(initial_unbridged_ds['MeasureId'])]

## Matching EDTA-nDS scores after single and double bridging

### Single bridging

In [ ]:
nds_bridged_df[nds_bridged_df['MeasureId'].isin(matched_edta_nds['MeasureId_nds'])]

In [ ]:
nds_bridged_postANML[nds_bridged_postANML['MeasureId'].isin(matched_edta_nds['MeasureId_nds'])]

In [ ]:
nds_bridged_postANML[aptamers]/nds_bridged_df[aptamers]

In [ ]:
nds_bridged_df_short = nds_bridged_df[['MeasureId', 'PROphetScore']]
nds_bridged_postANML_short = nds_bridged_postANML[['MeasureId', 'PROphetScore']]
nds_bridged_df_short.columns = ['MeasureId_nds', 'PROphetScore_nds']
nds_bridged_postANML_short.columns = ['MeasureId_nds', 'PROphetScore_nds']

In [ ]:
nds_bridged_df_short

In [ ]:
nds_bridged_postANML_short

In [ ]:
matched_edta_nds_raw = matched_edta_nds.copy()

In [ ]:
matched_edta_nds_single_bridging = pd.merge(matched_edta_nds, nds_bridged_df_short, how='inner', on='MeasureId_nds')
matched_edta_nds_postANML_double_bridging = pd.merge(matched_edta_nds, nds_bridged_postANML_short, how='inner', on='MeasureId_nds')

In [ ]:
matched_edta_nds_postANML_double_bridging

In [ ]:
nds_bridged_postANML_short

In [ ]:
edta_final_measurements = pd.read_csv('../../data/excels/20260126_adat_data_measurements.csv')

In [ ]:
edta_final_measurements = edta_final_measurements[edta_final_measurements['MeasureId'].isin(matched_edta_nds['MeasureId_edta'])]

In [ ]:
edta_final_measurements['PROphetScore_edta'] = get_prophet_score(edta_final_measurements, model)

In [ ]:
edta_final_measurements = edta_final_measurements[['MeasureId', 'PROphetScore_edta']]
edta_final_measurements.columns = ['MeasureId_edta', 'PROphetScore_edta']

In [ ]:
matched_edta_nds_single_bridging

In [ ]:
edta_nds = pd.merge(matched_edta_nds_single_bridging, edta_final_measurements, how='inner', on='MeasureId_edta')
edta_nds_postANML = pd.merge(matched_edta_nds_postANML_double_bridging, edta_final_measurements, how='inner', on='MeasureId_edta')

In [ ]:
edta_nds_postANML

In [ ]:
edta_nds[['PROphetScore_edta','PROphetScore_nds']]

In [ ]:
edta_nds_postANML #[['PROphetScore_edta','PROphetScore_nds']]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score


def scatter_with_regression(x, y,
                            xlabel="X",
                            ylabel="Y",
                            title="Scatter Plot with Linear Regression"):

    x = np.array(x)
    y = np.array(y)

    # Linear regression
    slope, intercept = np.polyfit(x, y, 1)

    # Regression line
    x_line = np.linspace(min(x), max(x), 100)
    y_line = slope * x_line + intercept

    # R²
    y_pred = slope * x + intercept
    r2 = r2_score(y, y_pred)

    # Plot
    plt.figure()
    plt.scatter(x, y, label="Data")

    # Regression line
    plt.plot(x_line, y_line,
             label=f"Regression: y = {slope:.3f}x + {intercept:.3f}")

    # X = Y reference line
    min_val = min(min(x), min(y))
    max_val = max(max(x), max(y))
    plt.plot([min_val, max_val], [min_val, max_val],
             linestyle=":",
             label="X = Y")

    # Equation text
    equation = f"R² = {r2:.3f}"
    plt.text(min(x), max(y), equation)

    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)

    plt.legend()

    plt.show()

    return slope, intercept, r2

In [ ]:
scatter_with_regression(edta_nds['PROphetScore_edta'], edta_nds['PROphetScore_nds'], xlabel="EDTA PROphet Score", ylabel="nDS PROphet Score", 
                        title="EDTA vs nDS PROphet Scores, prehybridization bridging only")

In [ ]:
scatter_with_regression(edta_nds_postANML['PROphetScore_edta'], edta_nds_postANML['PROphetScore_nds'], xlabel="EDTA PROphet Score", ylabel="nDS PROphet Score", 
                        title="EDTA vs nDS PROphet Scores, double bridging")

In [ ]:
confusion_double_bridging = pd.crosstab(edta_nds_postANML['PROphetScore_edta']>=5, edta_nds_postANML['PROphetScore_nds']>=5, rownames=['EDTA'], colnames=['nDS'])

In [ ]:
print(confusion_double_bridging)

In [ ]:
confusion_single_bridging = pd.crosstab(edta_nds['PROphetScore_edta']>=5, edta_nds['PROphetScore_nds']>=5, rownames=['EDTA'], colnames=['nDS'])

In [ ]:
print(confusion_single_bridging)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


def plot_confusion_matrix(conf, title="Confusion Matrix", ylabel="Reference", xlabel="Test", save_path=None):
    """
    Plot a presentation-ready confusion matrix heatmap.

    Parameters
    ----------
    conf : pandas.DataFrame
        2x2 confusion matrix from pd.crosstab
    title : str
        Plot title
    save_path : str or None
        If provided, saves the figure to this path

    Returns
    -------
    metrics : dict
        Dictionary with TP, TN, FP, FN and accuracy
    """

    # Ensure full 2x2 structure
    conf = conf.reindex(index=[False, True], columns=[False, True], fill_value=0)

    TN = conf.loc[False, False]
    FP = conf.loc[False, True]
    FN = conf.loc[True, False]
    TP = conf.loc[True, True]

    total = conf.values.sum()
    accuracy = (TP + TN) / total if total > 0 else 0

    plt.figure(figsize=(4,4))

    sns.heatmap(
        conf,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False
    )

    plt.title(f"{title}\nAgreement = {accuracy:.2%} (n={total})")
    plt.ylabel(ylabel)
    plt.xlabel(xlabel)

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300)

    plt.show()

    return {
        "TP": TP,
        "TN": TN,
        "FP": FP,
        "FN": FN,
        "accuracy": accuracy
    }

In [ ]:
plot_confusion_matrix(confusion_single_bridging, title="Single Bridging Confusion Matrix", xlabel='nDS', ylabel="EDTA", save_path="results/confusion_matrices/single_bridging_confusion.png")

In [ ]:
plot_confusion_matrix(confusion_double_bridging, title="Double Bridging Confusion Matrix",  xlabel='nDS', ylabel="EDTA", save_path="results/confusion_matrices/double_bridging_confusion.png")

## Matching EDTA-DS scores after single and double bridging

In [ ]:
matched_ds

In [ ]:
pd.merge(matched_edta, matched_ds, how='inner', on='SubjectId', suffixes=('_edta', '_ds'))['MeasureId_ds'].str.split('_').str[:2].str.join("_").value_counts()

# MDACC

In [ ]:
matched_mdacc_df = pd.read_excel('../../data/excels/matched MDACC proteomics with scores.xlsx')

In [ ]:
matched_mdacc_df

In [ ]:
matched_mdacc_df.columns

In [ ]:
matched_mdacc_df['plate'] = matched_mdacc_df['STRECK_MeasureId'].str.split('_').str[:2].str.join("_")

In [ ]:
matched_mdacc_df['plate'].value_counts()

In [ ]:
streck_matched_plates = matched_mdacc_df[matched_mdacc_df['plate'].str.startswith(
    'OH2026') | matched_mdacc_df['plate'].str.endswith(
        '052') | matched_mdacc_df['plate'].str.endswith(
            '054') | matched_mdacc_df['plate'].str.endswith('055')]['plate'].unique()

In [ ]:
streck_matched_plates

In [ ]:
streck_match_df = matched_mdacc_df[matched_mdacc_df['plate'].str.startswith(
    'OH2026') | matched_mdacc_df['plate'].str.endswith(
        '052') | matched_mdacc_df['plate'].str.endswith(
            '054') | matched_mdacc_df['plate'].str.endswith('055')]

In [ ]:
streck_match_df

In [ ]:
base_rfus.columns[:30]

In [ ]:
base_rfus['SampleType'].value_counts()

In [ ]:
info_df['SampleMatrixType'].value_counts()

In [ ]:
info_df = pd.read_csv('../../data/excels/20260309_adat_data_samples_with_sample_info.csv')

In [ ]:
info_df[info_df['PlateId']=='OH2026_004'][['MeasureId', 'SampleMatrixType', 'SubType']]

In [ ]:
info_df['SampleMatrixType'].value_counts()

In [ ]:
info_df[(info_df['PlateId']=='OH2026_009') & (info_df['SampleMatrixType']=='Streck non-DS')][['PlatePosition', 'SampleMatrixType']]

In [ ]:
matched_mdacc_df[matched_mdacc_df['plate'].str.startswith(
    'OH2026') | matched_mdacc_df['plate'].str.endswith(
        '052') | matched_mdacc_df['plate'].str.endswith('054')]

In [ ]:
mdacc_initial_adats_df = concat_initial_dfs('results/adats/madmed_for_MDACC_comparison')
mdacc_final_adats_df = concat_final_dfs('results/adats/madmed_for_MDACC_comparison')

In [ ]:
mdacc_initial_adats_df = add_measure_id(mdacc_initial_adats_df)
mdacc_final_adats_df = add_measure_id(mdacc_final_adats_df)

In [ ]:
mdacc_final_adats_df.head()

In [ ]:
mdacc_streck_final = mdacc_final_adats_df[mdacc_final_adats_df['MeasureId'].isin(streck_match_df['STRECK_MeasureId'])]

In [ ]:
mdacc_streck_final

In [ ]:
ds_bridged_postANML_coefficients

In [ ]:
with open('../../data/params/post_hybridization/ds_med_08032026.json', 'r') as fp:
    dummy = json.load(fp)

In [ ]:
for key in dummy.keys():
    if dummy[key] != ds_bridged_postANML_coefficients[key]:
        print(f"Mismatch for {key}: {dummy[key]} vs {ds_bridged_postANML_coefficients[key]}")

In [ ]:
mdacc_streck_final_transformed = transform_median_post_anml(mdacc_streck_final, ds_bridged_postANML_coefficients)

In [ ]:
mdacc_streck_final_transformed['PROphetScore'] = get_prophet_score(mdacc_streck_final_transformed, model)

In [ ]:
mdacc_streck_final['PROphetScore'] = get_prophet_score(mdacc_streck_final, model)

In [ ]:
mdacc_streck_final_transformed_short = mdacc_streck_final_transformed[['MeasureId', 'PROphetScore']]
mdacc_streck_final_short = mdacc_streck_final[['MeasureId', 'PROphetScore']]

In [ ]:
mdacc_streck_final_transformed_short.rename(columns={'MeasureId': 'STRECK_MeasureId', 'PROphetScore': 'PROphetScore_streck'})

In [ ]:
matches = matched_mdacc_df[matched_mdacc_df['STRECK_MeasureId'].isin(mdacc_streck_final_short['MeasureId'])][['STRECK_MeasureId', 'PROPHET_EDTA_MeasureId']]

In [ ]:
matches

In [ ]:
adat_measurements = pd.read_csv('../../data/excels/20260309_adat_data_measurements.csv')

In [ ]:
edta_final_measurements = adat_measurements[adat_measurements['MeasureId'].isin(matches['PROPHET_EDTA_MeasureId'])]

In [ ]:
edta_final_measurements['PROphetScore'] = get_prophet_score(edta_final_measurements, model)

In [ ]:
edta_final_measurements = pd.merge(matches, 
                                   edta_final_measurements[['MeasureId', 'PROphetScore']],
                                   how='inner',
                                   left_on='PROPHET_EDTA_MeasureId',
                                   right_on='MeasureId')[['STRECK_MeasureId', 'PROPHET_EDTA_MeasureId', 'PROphetScore']]

In [ ]:
edta_final_measurements = edta_final_measurements.rename(columns={'PROphetScore': 'PROphetScore_edta'})

In [ ]:
pd.merge(edta_final_measurements,mdacc_streck_final_transformed_short, 

In [ ]:
edta_final_measurements.head(1)

In [ ]:
mdacc_streck_final_transformed_short.head(1)

In [ ]:
mdacc_streck_final_transformed_short.rename(columns={'MeasureId': 'STRECK_MeasureId', 'PROphetScore': 'PROphetScore_streck'}, inplace=True)

In [ ]:
mdacc_ds_comparison = pd.merge(edta_final_measurements, mdacc_streck_final_transformed_short, how='inner', on='STRECK_MeasureId')[['PROphetScore_edta', 'PROphetScore_streck']]

In [ ]:
confusion_double_bridging = pd.crosstab(edta_nds_postANML['PROphetScore_edta']>=5, edta_nds_postANML['PROphetScore_nds']>=5, rownames=['EDTA'], colnames=['nDS'])

In [ ]:
mdacc_ds_comparison.head(1)

In [ ]:
confusion_mdacc_comparison = pd.crosstab(mdacc_ds_comparison['PROphetScore_edta']>=5, mdacc_ds_comparison['PROphetScore_streck']>=5, rownames=['EDTA'], colnames=['STRECK'])

In [ ]:
plot_confusion_matrix(confusion_mdacc_comparison, title="MDACC DS-EDTA Confusion Matrix", xlabel='STRECK', ylabel="EDTA", save_path="results/confusion_matrices/mdacc_comparison_confusion.png")

In [ ]:
scatter_with_regression(mdacc_ds_comparison['PROphetScore_edta'], mdacc_ds_comparison['PROphetScore_streck'], xlabel="EDTA PROphet Score", ylabel="STRECK DS PROphet Score", 
                        title="EDTA vs DS PROphet Scores, double bridging")

## BOOM

In [ ]:
ds_unbridged_df['PROphetScore'] = get_prophet_score(ds_unbridged_df, model)
ds_bridged_df['PROphetScore'] = get_prophet_score(ds_bridged_df, model)
nds_unbridged_df['PROphetScore'] = get_prophet_score(nds_unbridged_df, model)
nds_bridged_df['PROphetScore'] = get_prophet_score(nds_bridged_df, model)

In [ ]:
overlay_histograms([ds_unbridged_df['PROphetScore'], ds_bridged_df['PROphetScore'], nds_unbridged_df['PROphetScore'], nds_bridged_df['PROphetScore']],
                 ['DS Unbridged', 'DS Bridged', 'nDS Unbridged', 'nDS Bridged']  )

In [ ]:
ds_unbridged_df = get_prophet_score(ds_unbridged_df, model)
ds_bridged_df = get_prophet_score(ds_bridged_df, model)
nds_unbridged_df = get_prophet_score(nds_unbridged_df, model)
nds_bridged_df = get_prophet_score(nds_bridged_df, model)

In [ ]:
ds_bridged_df = bridged_df[bridged_df['MeasureId'].isin(ds_rfu_df['MeasureId'])]
ds_unbridged_df = unbridged_df[unbridged_df['MeasureId'].isin(ds_rfu_df['MeasureId'])]

nds_bridged_df = bridged_df[bridged_df['MeasureId'].isin(nds_rfu_df['MeasureId'])]
nds_unbridged_df = unbridged_df[unbridged_df['MeasureId'].isin(nds_rfu_df['MeasureId'])]

In [ ]:
ds_bridged_df['RowCheck'].value_counts()

In [ ]:
ds_unbridged_df['RowCheck'].value_counts()

In [ ]:
nds_bridged_df['RowCheck'].value_counts()

In [ ]:
nds_unbridged_df['RowCheck'].value_counts()

In [ ]:
ds_bridged_df['ProphetScore'] = get_prophet_score(ds_bridged_df, model)
ds_unbridged_df['ProphetScore'] = get_prophet_score(ds_unbridged_df, model)
nds_bridged_df['ProphetScore'] = get_prophet_score(nds_bridged_df, model)
nds_unbridged_df['ProphetScore'] = get_prophet_score(nds_unbridged_df, model)

In [ ]:
ds_bridged_df['ProphetScore'].hist(alpha=0.5, label='Bridged DS')

In [ ]:
ds_unbridged_df['ProphetScore'].hist(alpha=0.5, label='Unbridged DS')

In [ ]:
nds_bridged_df['ProphetScore'].hist(alpha=0.5, label='Bridged nDS')

In [ ]:
nds_unbridged_df['ProphetScore'].hist(alpha=0.5, label='Unbridged nDS')

In [ ]:
plot_overlaid_histograms(ds_bridged_df['ProphetScore'], ds_unbridged_df['ProphetScore'], 'Bridged DS Prophet Scores', 'Unbridged DS Prophet Scores', 'DS Bridged vs Unbridged Prophet Score Distribution')

# Assessing initial distributions

In [ ]:
ds_rfu_df

In [ ]:
scatter_comparison(edta_rfu_df[aptamers], 
                       ds_rfu_051_003_df[aptamers], 
                       metric_type='mean',
                       xlabel='EDTA', 
                       ylabel='Streck', 
                       title='Mean comparison of pre-normalization RFUs, no bridging',
                       trim_percentile=100)

In [ ]:
scatter_comparison(edta_rfu_df[aptamers], 
                       ds_rfu_051_003_df[aptamers], 
                       metric_type='median',
                       xlabel='EDTA', 
                       ylabel='Streck', 
                       title='Median comparison of pre-normalization RFUs, no bridging',
                       trim_percentile=100)

In [ ]:
scatter_comparison(edta_rfu_df[aptamers], 
                       ds_rfu_051_003_df[aptamers], 
                       metric_type='median',
                       xlabel='EDTA', 
                       ylabel='Streck', 
                       title='Median comparison of pre-normalization RFUs, no bridging',
                       trim_percentile=100)

In [ ]:
scatter_comparison(edta_rfu_df[aptamers], 
                       ds_rfu_051_003_df[aptamers], 
                       metric_type='std',
                       xlabel='EDTA', 
                       ylabel='Streck', 
                       title='Standard deviation comparison of pre-normalization RFUs, no bridging',
                       trim_percentile=100)

In [ ]:
scatter_comparison(edta_rfu_df[aptamers], 
                       streck_rfu_df[aptamers], 
                       metric_type='mad',
                       xlabel='EDTA', 
                       ylabel='Streck', 
                       title='Median absolute deviation comparison of pre-normalization RFUs, no bridging',
                       trim_percentile=100)

# README on running SL pipeline

# Reading ADATs and assessing initial bridged values

In [ ]:
import os
adat_path = 'data/adats/'

initial_adats = []
final_adats = []
for folder in os.listdir(adat_path):
    splits = folder.split('_')
    initial_file = splits[0] + '_' + splits[1] + '.adat'
    folder_path = os.path.join(adat_path, folder)
    files = os.listdir(folder_path)
    final_file = max(files, key=len)
    initial_adat, _ = read_adat_file(os.path.join(folder_path, initial_file))
    final_adat, _ = read_adat_file(os.path.join(folder_path, final_file))
    initial_adats.append(initial_adat)
    final_adats.append(final_adat)


In [ ]:
initial_bridged_df = add_measure_id(pd.concat(initial_adats, ignore_index=True))
final_bridged_df = add_measure_id(pd.concat(final_adats, ignore_index=True))

In [ ]:
initial_bridged_df

In [ ]:
ds_rfu_051_003_df['MeasureId']

In [ ]:
initial_bridged_ds_df = initial_bridged_df[initial_bridged_df['MeasureId'].isin(ds_rfu_051_003_df['MeasureId'])]
initial_bridged_nds_df = initial_bridged_df[initial_bridged_df['MeasureId'].isin(nds_rfu_df['MeasureId'])]
final_bridged_ds_df = final_bridged_df[final_bridged_df['MeasureId'].isin(ds_rfu_051_003_df['MeasureId'])]
final_bridged_nds_df = final_bridged_df[final_bridged_df['MeasureId'].isin(nds_rfu_df['MeasureId'])]

In [ ]:
final_bridged_ds_df.shape

In [ ]:
nds_rfu_df['MeasureId']

In [ ]:
scatter_comparison(edta_rfu_df[aptamers], 
                       initial_bridged_nds_df[aptamers], 
                       metric_type='median',
                       xlabel='EDTA', 
                       ylabel='Non-desalinated Streck', 
                       title='Standard Median comparison of pre-normalization RFUs, bridged',
                       trim_percentile=100)

In [ ]:
(edta_rfu_df[aptamers].median() / initial_bridged_nds_df[aptamers].median()).mean()

In [ ]:
scatter_comparison(np.log(edta_rfu_df[aptamers]), 
                       np.log(initial_bridged_nds_df[aptamers]), 
                       metric_type='mad',
                       xlabel='EDTA', 
                       ylabel='Non-desalinated Streck', 
                       title='Standard MAD comparison of pre-normalization RFUs, bridged',
                       trim_percentile=100)

In [ ]:
scatter_comparison(np.log(edta_rfu_df[aptamers]), 
                       np.log(initial_bridged_ds_df[aptamers]), 
                       metric_type='median',
                       xlabel='EDTA', 
                       ylabel='Desalinated Streck', 
                       title='Standard Median comparison of pre-normalization RFUs, bridged',
                       trim_percentile=100)

In [ ]:
with open('../../data/conversion_coefficients/MAD_median_nds_coefficients_plates_OH2025_054_OH2025_055_OH2026_001_OH2026_002_04022026.json', 'r') as fp:
    params_nds = json.load(fp)

In [ ]:
params_nds

In [ ]:
import pandas as pd
import numpy as np

def transform_columns(df, transform_dict):
    """
    Transform specified columns using the formula:
    new_value = (e^b) * (old_value^a)
    where a = med_e/med_s and b = mad_e - (med_e/med_s)*mad_s
    
    Parameters:
    df: DataFrame to transform
    transform_dict: Dictionary with column names as keys and stats as values
    
    Returns:
    DataFrame with transformed columns
    """
    df_transformed = df.copy()
    
    for col_name, stats in transform_dict.items():
        if col_name in df_transformed.columns:
            # Calculate transformation parameters
            a = stats['med_edta'] / stats['med_streck']
            b = stats['mad_edta'] - (stats['med_edta'] / stats['med_streck']) * stats['mad_streck']
            
            # Apply transformation: new_value = (e^b) * (old_value^a)
            df_transformed[col_name] = (np.exp(b)) * (df_transformed[col_name] ** a)
    
    return df_transformed

# Example usage:
# df_new = transform_columns(df, your_dict)

In [ ]:
nds_rfu_df.shape

In [ ]:
transformed_nds = transform_columns(nds_rfu_df, params_nds)

In [ ]:
transformed_nds

In [ ]:
scatter_comparison(np.log(edta_rfu_df[aptamers]), 
                       np.log(initial_bridged_nds_df[aptamers]), 
                       metric_type='median',
                       xlabel='EDTA', 
                       ylabel='Desalinated Streck', 
                       title='Standard Median comparison of pre-normalization RFUs, bridged',
                       trim_percentile=100)

In [ ]:
scatter_comparison(np.log(edta_rfu_df[aptamers]), 
                       np.log(transformed_nds[aptamers]), 
                       metric_type='median',
                       xlabel='EDTA', 
                       ylabel='Desalinated Streck', 
                       title='Standard Median comparison of pre-normalization RFUs, bridged',
                       trim_percentile=100)

# PROphet score comparison

In [ ]:
model = pd.read_pickle('../../data/models/DCB_nosex_gamma1_rescaled.pkl')

In [ ]:
final_bridged_ds_df['PROphetScore'] = get_prophet_score(final_bridged_ds_df, model)
final_bridged_nds_df['PROphetScore'] = get_prophet_score(final_bridged_nds_df, model)

In [ ]:
final_edta_df = pd.read_csv('data/20260208_adat_data_measurements.csv')

In [ ]:
final_edta_df = final_edta_df[final_edta_df['MeasureId'].isin(edta_rfu_df['MeasureId'])]

In [ ]:
final_edta_df.shape

In [ ]:
final_edta_df['PROphetScore'] = get_prophet_score(final_edta_df, model)

In [ ]:
plot_overlaid_histograms(final_edta_df['PROphetScore'], final_bridged_ds_df['PROphetScore'], 10, xtitle='EDTA PROphet score', ytitle='Desalted Streck PROphet score', trim_percentile=100)


In [ ]:
plot_overlaid_histograms(final_edta_df['PROphetScore'], final_bridged_nds_df['PROphetScore'], 10, xtitle='EDTA PROphet score', ytitle='Non-desalted Streck PROphet score', trim_percentile=100)


In [ ]:
(final_bridged_nds_df['PROphetScore'] >= 5).sum()

In [ ]:
(final_bridged_ds_df['PROphetScore'] >= 5).sum()

In [ ]:
final_bridged_nds_df['PROphetScore'].mean()

In [ ]:
final_bridged_ds_df['PROphetScore'].mean()

In [ ]:
nds_rfu_df.shape

In [ ]:
nds_rfu_df[['MeasureId', 'PlateId', 'PlatePosition', 'PlateRunDate']].to_csv('streck_non_desalted_samples.csv')

In [ ]:
nds_rfu_df.shape

In [ ]:
ds_rfu_df[['MeasureId', 'PlateId', 'PlatePosition', 'PlateRunDate']].to_csv('streck_desalted_samples.csv')

In [ ]:
ds_rfu_df[~ds_rfu_df['PlateId'].isin(['OH2025_049', 'OH2025_050'])][['MeasureId', 'PlateId', 'PlatePosition', 'PlateRunDate']].reset_index(drop=True).to_csv('streck_desalted_samples.csv')

In [ ]:
ds_rfu_df[~ds_rfu_df['PlateId'].isin(['OH2025_049', 'OH2025_050'])][['MeasureId', 'PlateId', 'PlatePosition', 'PlateRunDate']]

In [ ]:
initial_df = pd.concat(initial_adats, ignore_index=True)
final_df = pd.concat(final_adats, ignore_index=True)                    

In [ ]:
scatter_comparison(edta_rfu_df[aptamers], 
                       initial_df[aptamers], 
                       metric_type='median',
                       xlabel='EDTA', 
                       ylabel='Streck', 
                       title='Median comparison of pre-normalization RFUs, after bridging',
                       trim_percentile=100)

In [ ]:
scatter_comparison(edta_rfu_df[aptamers], 
                       initial_df[aptamers], 
                       metric_type='mad',
                       xlabel='EDTA', 
                       ylabel='Streck', 
                       title='MAD comparison of pre-normalization RFUs, after bridging',
                       trim_percentile=100)

In [ ]:
scatter_comparison(edta_rfu_df[aptamers], 
                       final_df[aptamers], 
                       metric_type='median',
                       xlabel='EDTA', 
                       ylabel='Streck', 
                       title='Median comparison of post-normalization RFUs, after bridging',
                       trim_percentile=90)

# Comparing distributions